# Hybrid LSTM–XGBoost forecasting of hourly industrial electricity use
Runs in Google Colab or Jupyter. Without your own data it uses the synthetic files created in step 3.

Outputs:
1. Model comparison – persistence, seasonal naive, LSTM, XGBoost, reciprocal-error weighting (Chen et al., 2019), stacked hybrid (mean ± SD over seeds)
2. Ablation – no shift split, no temperature anomaly, no LSTM meta-feature
3. Diebold–Mariano and Wilcoxon tests
4. Expanding-window time-series cross-validation of the XGBoost meta-learner
5. Peak-load reduction rule on **measured** load, triggered by the forecast (5/10/15 %), with flagging accuracy and threshold sensitivity
6. Daily check against utility meter registers (optional)
7. Figures (PNG and TIFF, 300 dpi)

**Protocol:** chronological 80/20 split; Min–Max scaling and monthly temperature from the training period only.

## 1. Setup

In [ ]:
import os, sys, subprocess, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
os.makedirs("data", exist_ok=True)
print("Working folder:", os.getcwd())

## 2. Settings
To use your own data, set `HOURLY` (and optionally `REPORTS`, `PRODUCTION`) to your files. `QUICK = True` gives a short smoke test (1 seed, 2 epochs).

In [ ]:
HOURLY = "data/plant_hourly.csv"        # your hourly file (see README for the columns)
REPORTS = "reports/*.xlsx"               # optional: monthly utility meter reports
PRODUCTION = "data/daily_production.csv" # optional: daily production
TRIAL_DATE = "2025-08-06"                # date on which the measure was first trialled (None to skip)
THRESHOLD = 2200.0
QUICK = True

OUT, OUT_PEAK, OUT_METER, OUT_FIG = "results", "results_peak", "results_meter", "figures"
USE_OWN = os.path.exists(HOURLY)
USE_REPORTS = len(glob.glob(REPORTS)) > 0
print("Own hourly file:", USE_OWN, "| meter reports:", USE_REPORTS)

## 3. Synthetic demo files

In [ ]:
if not (USE_OWN and USE_REPORTS):
    import argparse
    import numpy as np
    import pandas as pd

    ap = argparse.ArgumentParser()
    ap.add_argument("--start", default="2022-01-01 00:00")
    ap.add_argument("--end", default="2023-08-25 09:00", help="last hour of the synthetic series")
    ap.add_argument("--hours", type=int, default=10000,
                    help="number of hourly rows to keep; the rest are removed as random gaps (like meter outages)")
    ap.add_argument("--test-start", default="2023-04-26 07:00",
                    help="optional timestamp of the first test hour; gaps are placed so that the last 20 %% of rows start here")
    a, _ = ap.parse_known_args()   # tolerant of notebook kernels

    rng = np.random.default_rng(2026)
    t = pd.date_range(a.start, a.end, freq="h")
    n = len(t)
    h, m, dow = t.hour.values, t.month.values, t.dayofweek.values
    season = np.where(np.isin(m, [1, 2, 3]), "winter", np.where(np.isin(m, range(4, 9)), "spring", "autumn"))
    temp = 26.8 + 3 * np.sin(2 * np.pi * (m - 4) / 12) + 3 * np.sin(2 * np.pi * (h - 9) / 24) + rng.normal(0, 1, n)
    rh = np.clip(70 - 1.5 * (temp - 26.8) + rng.normal(0, 8, n), 30, 100)
    campaign = 1 + 0.5 * np.isin(m, [6, 7, 10])                      # seasonal campaign months
    work = ((h >= 8) & (h < 20) & (dow < 6)).astype(float)
    process = np.clip(rng.normal(15000, 300, n) * (0.98 + 0.04 * work), 8000, 20000)
    paste = np.clip(rng.normal(1500, 400, n) * work * campaign, 0, None)
    cold = np.clip(rng.normal(1000, 900, n), 500, 15000)
    shut = rng.random(n) < (0.015 + 0.01 * ((h >= 19) | (h < 6)))   # unplanned stops occur mostly at night
    # observed drivers (production, schedule, weather)
    base = 1500 * campaign * (1 + 0.03 * work) + 8 * (temp - 26.8) + 0.03 * paste   # plant profile is almost flat over the day
    # unobserved refrigeration state (compressors, defrost, rooms in use): a persistent AR(1) component,
    # so that consumption depends on the preceding hours as in the plant data
    z = np.zeros(n)
    e = rng.normal(0, 1, n)
    for i in range(1, n):
        z[i] = 0.97 * z[i - 1] + e[i]
    night = ((h >= 19) | (h < 6)).astype(float)
    # the plant load is more variable at night (persistence error is about 1.3 times the day-shift value)
    raw = base + 120 * z + rng.normal(0, 150, n) * (1 + 0.6 * night)
    raw = pd.Series(raw).ewm(alpha=0.6).mean().to_numpy()
    # rescale to a typical plant load level (mean about 2,090 kWh, SD about 850 kWh)
    en = 2090 + 850 * (raw - raw.mean()) / raw.std()
    en[shut] = rng.uniform(20, 400, shut.sum())
    en = np.clip(en, 18, 5200)
    df = pd.DataFrame({"timestamp": t.strftime("%d/%m/%Y %H:%M"), "EN": en.round(2), "CL": (0.75 * en).round(4),
                       "RH": rh.round(1), "Air temp": temp.round(2), "Process": process.round(2),
                       "Paste": paste.round(0), "Cold": cold.round(0), "season": season})
    # remove random blocks of hours (2-48 h) so the file has gaps like a real meter log
    keep = np.ones(n, bool)
    g = np.random.default_rng(7)
    def drop_gaps(lo, hi, k):
        """remove k hours as random 2-48 h blocks inside rows [lo, hi)"""
        while (~keep[lo:hi]).sum() < k:
            s0 = int(g.integers(lo, hi))
            idx = np.arange(s0, min(s0 + int(g.integers(2, 49)), hi))
            idx = idx[keep[idx]][: k - (~keep[lo:hi]).sum()]
            keep[idx] = False
    if a.test_start is None:
        drop_gaps(1, n - 1, max(0, n - a.hours))
    else:
        cut = int(np.searchsorted(t, pd.Timestamp(a.test_start)))
        n_test = a.hours - int(a.hours * 0.8)
        drop_gaps(1, cut, max(0, cut - int(a.hours * 0.8)))
        drop_gaps(cut + 1, n - 1, max(0, (n - cut) - n_test))
    df = df[keep].reset_index(drop=True)
    df.to_csv("data/synthetic_hourly.csv", index=False)
    print("written", len(df), "rows,", df.timestamp.iloc[0], "to", df.timestamp.iloc[-1])

    # ---- synthetic 2025 daily meter registers and daily production (for daily_meter_check.py)
    days = pd.date_range("2025-06-01", "2025-10-31", freq="D")
    th = __import__("holidays").Thailand(years=2025)
    out = np.clip(rng.normal(150000, 60000, len(days)), 9000, 420000)
    kwh = 45000 + 0.15 * out + rng.normal(0, 6000, len(days))
    work = np.array([(d.dayofweek < 5) and (d.date() not in th) for d in days])
    rate_a = np.where(work, kwh * rng.uniform(0.40, 0.52, len(days)), np.nan)
    rate_b = np.where(work, kwh - np.nan_to_num(rate_a), np.nan)
    rate_c = np.where(work, np.nan, kwh)
    pd.DataFrame({"date": days.date, "kWh_rateA": rate_a.round(2), "kWh_rateB": rate_b.round(2),
                  "kWh_rateC": rate_c.round(2), "kWh_total": kwh.round(2)}).to_csv("data/synthetic_daily_meter.csv", index=False)
    share = rng.dirichlet([3, 3, 4], len(days))
    pd.DataFrame({"date": days.date, "paste_output_kg": (out * share[:, 0]).round(1), "packing_output_kg": (out * share[:, 1]).round(1),
                  "freezing_output_kg": (out * share[:, 2]).round(1), "total_output_kg": out.round(1)}).to_csv("data/synthetic_daily_production.csv", index=False)
    print("written synthetic 2025 daily files")


In [ ]:
DATA = HOURLY if USE_OWN else "data/synthetic_hourly.csv"
print("Hourly data used:", DATA)

## 4. Libraries and model settings

In [ ]:
import argparse
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import holidays  # noqa: E402
import tensorflow as tf  # noqa: E402
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit  # noqa: E402
from sklearn.preprocessing import MinMaxScaler  # noqa: E402
from xgboost import XGBRegressor  # noqa: E402

# ----------------------------------------------------------------------------- settings
SEQ_LEN = 24
LSTM_UNITS = 64
LSTM_EPOCHS = 20
LSTM_BATCH = 32
LSTM_VAL_SPLIT = 0.10
XGB_PARAMS = dict(n_estimators=100, max_depth=6, learning_rate=0.10)
SEEDS = [42, 0, 1, 2, 3]
TRAIN_FRACTION = 0.80
DAY_HOURS = range(6, 19)            # 06:00-18:59 = day shift
ON_PEAK_HOURS = range(9, 22)        # weekday TOU On-Peak 09:00-21:59
THRESHOLD_KWH = 2200.0              # development-period peak-load threshold
ALPHAS = [0.05, 0.10, 0.15]
TARIFF_THB = 5.0
CV_GRID = dict(n_estimators=[100, 200, 300], max_depth=[3, 4, 6, 8], learning_rate=[0.05, 0.10])

FEATURES = [
    "RH", "air_temp", "monthly_avg_temp", "temp_anomaly",
    "Process", "Paste", "Cold",
    "hour", "month", "dow", "daytime", "on_peak", "holiday",
    "autumn", "spring", "winter",
    "lag1", "lag24",
]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


class Log:
    def __init__(self, path):
        self.f = open(path, "w", encoding="utf-8")

    def __call__(self, msg=""):
        print(msg)
        self.f.write(str(msg) + "\n")
        self.f.flush()

## 5. Data preparation

In [ ]:
# ----------------------------------------------------------------------------- data
def load_plant_file(path, impute="mean_train"):
    raw = pd.read_csv(path, encoding="utf-8-sig")
    raw.columns = [c.strip() for c in raw.columns]
    raw = raw.dropna(subset=["timestamp"]).copy()
    raw["dt"] = pd.to_datetime(raw["timestamp"].astype(str).str.strip(), dayfirst=True, errors="coerce")
    raw = raw.dropna(subset=["dt"]).sort_values("dt").drop_duplicates("dt").reset_index(drop=True)
    d = pd.DataFrame({"dt": raw["dt"], "EN": raw["EN"], "RH": raw["RH"], "air_temp": raw["Air temp"],
                      "Process": raw["Process"], "Paste": raw["Paste"], "Cold": raw["Cold"],
                      "season": raw["season"].astype(str).str.strip().str.lower()})
    n = len(d)
    cut = int(n * TRAIN_FRACTION)
    train_mask = np.arange(n) < cut
    missing = d.isna().sum()
    num = ["EN", "RH", "air_temp", "Process", "Paste", "Cold"]
    if impute == "interpolate":
        d[num] = d.set_index("dt")[num].interpolate(method="time").bfill().ffill().values
    else:  # training-period means only
        d[num] = d[num].fillna(d.loc[train_mask, num].mean())
    t = d["dt"]
    th = holidays.Thailand(years=sorted(t.dt.year.unique().tolist()))
    d["holiday"] = t.dt.date.map(lambda x: int(x in th))
    d["hour"] = t.dt.hour
    d["month"] = t.dt.month
    d["dow"] = t.dt.dayofweek
    d["daytime"] = t.dt.hour.isin(DAY_HOURS).astype(int)
    d["on_peak"] = ((d["dow"] < 5) & d["hour"].isin(ON_PEAK_HOURS) & (d["holiday"] == 0)).astype(int)
    # monthly mean temperature from the training period only
    mtemp = d.loc[train_mask].groupby("month")["air_temp"].mean()
    d["monthly_avg_temp"] = d["month"].map(mtemp).fillna(d.loc[train_mask, "air_temp"].mean())
    d["temp_anomaly"] = d["air_temp"] - d["monthly_avg_temp"]
    for s in ["autumn", "spring", "winter"]:
        d[s] = (d["season"] == s).astype(int)
    # lags on the true hourly clock (a gap gives the last available value)
    s = d.set_index("dt")["EN"]
    d["lag1"] = s.shift(1, freq="h").reindex(d["dt"]).values
    d["lag24"] = s.shift(24, freq="h").reindex(d["dt"]).values
    d["lag1"] = d["lag1"].fillna(d["EN"].shift(1))
    d["lag24"] = d["lag24"].fillna(d["EN"].shift(24))
    return d, cut, missing


class Data:
    def __init__(self, d, cut):
        self.full = d
        self.target_full = d["EN"].values.astype(float)
        self.scaler = MinMaxScaler().fit(self.target_full[:cut].reshape(-1, 1))
        s = self.scaler.transform(self.target_full.reshape(-1, 1)).ravel()
        n = len(d)
        self.Xseq = np.stack([s[i - SEQ_LEN:i] for i in range(SEQ_LEN, n)])[..., None]
        self.df = d.iloc[SEQ_LEN:].reset_index(drop=True)
        self.y = self.df["EN"].values.astype(float)
        self.y_s = s[SEQ_LEN:]
        self.train = (np.arange(SEQ_LEN, n) < cut)
        self.test = ~self.train
        self.day = self.df["daytime"].values == 1

## 6. Models

In [ ]:
# ----------------------------------------------------------------------------- models
def build_lstm():
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(SEQ_LEN, 1)),
                             tf.keras.layers.LSTM(LSTM_UNITS, activation="relu"),
                             tf.keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    return m


def groups(D, split):
    return [(1, D.day), (0, ~D.day)] if split else [("all", np.ones(len(D.y), bool))]


def lstm_pred(D, seed, split=True, fit=None):
    fit = D.train if fit is None else fit
    out = np.zeros(len(D.y))
    for _, g in groups(D, split):
        set_seed(seed)
        m = build_lstm()
        m.fit(D.Xseq[fit & g], D.y_s[fit & g], epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH,
              validation_split=LSTM_VAL_SPLIT, verbose=0, shuffle=True)
        out[g] = m.predict(D.Xseq[g], verbose=0).ravel()
    return D.scaler.inverse_transform(out.reshape(-1, 1)).ravel()


def xgb_pred(D, feats, seed, split=True, meta=None, fit=None, params=None):
    fit = D.train if fit is None else fit
    X = D.df[feats].copy()
    if meta is not None:
        X["lstm_pred"] = meta
    out = np.zeros(len(D.y))
    for _, g in groups(D, split):
        m = XGBRegressor(random_state=seed, n_jobs=-1, **(params or XGB_PARAMS))
        m.fit(X[fit & g], D.y[fit & g])
        out[g] = m.predict(X[g])
    return out

## 7. Metrics and statistical tests

In [ ]:
# ----------------------------------------------------------------------------- metrics & tests
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    e = y - p
    nz = np.abs(y) > 1e-9
    return dict(RMSE=float(np.sqrt(np.mean(e ** 2))), MAE=float(np.mean(np.abs(e))),
                MAPE=float(np.mean(np.abs(e[nz] / y[nz])) * 100),
                R2=float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)))


def dm_test(e1, e2, h=1, hac_lags=None):
    """Diebold-Mariano, squared-error loss, HLN correction. Negative stat: model 1 more accurate."""
    d = e1 ** 2 - e2 ** 2
    n = len(d)
    dbar = d.mean()
    L = (h - 1) if hac_lags is None else hac_lags
    var = np.mean((d - dbar) ** 2)
    for k in range(1, L + 1):
        w = 1.0 if hac_lags is None else 1 - k / (L + 1)
        var += 2 * w * np.mean((d[k:] - dbar) * (d[:-k] - dbar))
    stat = dbar / np.sqrt(var / n) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    return float(stat), float(2 * stats.t.sf(abs(stat), df=n - 1))


def shift_table(D, preds, mask):
    rows = []
    for name, p in preds.items():
        for sh, g in (("All", np.ones(len(D.y), bool)), ("Day", D.day), ("Night", ~D.day)):
            m = mask & g
            rows.append(dict(Model=name, Shift=sh, **metrics(D.y[m], p[m])))
    return pd.DataFrame(rows)

## 8. Full experiment

In [ ]:
# ----------------------------------------------------------------------------- main
def main(data, out, quick=False):
    a = argparse.Namespace(data=data, out=out, quick=quick)
    global LSTM_EPOCHS, SEEDS, CV_GRID
    if a.quick:
        LSTM_EPOCHS, SEEDS = 2, [42]
        CV_GRID = dict(n_estimators=[100], max_depth=[3, 6], learning_rate=[0.10])
    os.makedirs(a.out, exist_ok=True)
    log = Log(os.path.join(a.out, "summary.txt"))
    t0 = time.time()

    d, cut, missing = load_plant_file(a.data)
    D = Data(d, cut)
    log("=" * 78)
    log("Hybrid LSTM-XGBoost - reproducibility run")
    log("=" * 78)
    log(f"Data: {len(d)} hourly rows, {d.dt.min()} to {d.dt.max()}")
    log(f"Training: {d.dt.iloc[0]} to {d.dt.iloc[cut - 1]} ({cut} rows)")
    log(f"Testing : {d.dt.iloc[cut]} to {d.dt.iloc[-1]} ({len(d) - cut} rows)")
    log(f"Rows used after the 24-h warm-up: train {D.train.sum()}, test {D.test.sum()}")
    log("Missing values in the input file: " + (", ".join(f"{k}={v}" for k, v in missing.items() if v) or "none"))
    ytr = D.y[D.train]
    log(f"EN training period: mean {ytr.mean():.1f}, max {ytr.max():.1f}, min {ytr.min():.1f} kWh")
    log(f"Test hours below 500 kWh (inflate MAPE): {(D.y[D.test] < 500).mean() * 100:.1f} %")

    feats = FEATURES
    feats_noanom = [f for f in feats if f != "temp_anomaly"]
    tr_idx = np.where(D.train)[0]
    fit90 = D.train & (np.arange(len(D.y)) < tr_idx[int(len(tr_idx) * 0.9)])
    val10 = D.train & ~fit90

    # ------------------------------------------------ models over seeds
    log("\n[1] Models over seeds " + str(SEEDS))
    tabs, errs, keep, lp_seed = [], {}, {}, {}
    for seed in SEEDS:
        log(f"  seed {seed}")
        lp = lstm_pred(D, seed, split=True)
        lp_seed[seed] = lp
        xo = xgb_pred(D, feats, seed, split=True)
        hy = xgb_pred(D, feats, seed, split=True, meta=lp)
        lp_all = lstm_pred(D, seed, split=False)
        a_nosplit = xgb_pred(D, feats, seed, split=False, meta=lp_all)
        a_noanom = xgb_pred(D, feats_noanom, seed, split=True, meta=lp)
        l90 = lstm_pred(D, seed, split=True, fit=fit90)
        x90 = xgb_pred(D, feats, seed, split=True, fit=fit90)
        chen = np.zeros(len(D.y))
        for sh, g in ((1, D.day), (0, ~D.day)):
            el = metrics(D.y[val10 & g], l90[val10 & g])["MAPE"]
            ex = metrics(D.y[val10 & g], x90[val10 & g])["MAPE"]
            w = (1 / el) / (1 / el + 1 / ex)
            chen[g] = w * lp[g] + (1 - w) * xo[g]
            if seed == SEEDS[0]:
                log(f"    Chen weights ({'day' if sh else 'night'}): LSTM {w:.3f}, XGBoost {1 - w:.3f}")
        P = {"Persistence (lag-1)": D.df["lag1"].values, "Seasonal naive (lag-24)": D.df["lag24"].values,
             "LSTM": lp, "XGBoost": xo, "Chen et al. weighting": chen, "Hybrid LSTM-XGBoost": hy,
             "Ablation: no shift split": a_nosplit, "Ablation: no temperature anomaly": a_noanom,
             "Ablation: no LSTM meta-feature": xo}
        t = shift_table(D, P, D.test)
        t["seed"] = seed
        tabs.append(t)
        errs[seed] = {k: D.y - v for k, v in P.items()}
        if seed == SEEDS[0]:
            keep = dict(lstm=lp, hybrid=hy)
            pred_out = pd.DataFrame({"dt": D.df["dt"], "measured": D.y, "shift": np.where(D.day, "day", "night"),
                                     **{k: v for k, v in P.items() if not k.startswith("Ablation")}})[D.test]
            pred_out.to_csv(os.path.join(a.out, "test_predictions_seed42.csv"), index=False)

    res = pd.concat(tabs)
    res.to_csv(os.path.join(a.out, "all_seeds_metrics.csv"), index=False)
    agg = res.groupby(["Model", "Shift"], sort=False)[["RMSE", "MAE", "MAPE", "R2"]].agg(["mean", "std"]).fillna(0.0)
    agg.to_csv(os.path.join(a.out, "table_mean_sd.csv"))

    def cell(r, c, d1):
        d = 3 if c == "R2" else d1
        m, sd = r[(c, "mean")], r[(c, "std")]
        return f"{m:.{d}f}" if sd == 0 else f"{m:.{d}f} ± {sd:.{d}f}"

    def fmt(name, d1=2, shifts=("All", "Day", "Night")):
        rows = []
        for sh in shifts:
            r = agg.loc[(name, sh)]
            rows.append(dict(Model=name, Shift=sh, **{c: cell(r, c, d1) for c in ("RMSE", "MAE", "MAPE", "R2")}))
        return rows

    main_models = ["Persistence (lag-1)", "Seasonal naive (lag-24)", "LSTM", "XGBoost",
                   "Chen et al. weighting", "Hybrid LSTM-XGBoost"]
    abl_models = ["Hybrid LSTM-XGBoost", "Ablation: no shift split", "Ablation: no temperature anomaly",
                  "Ablation: no LSTM meta-feature"]
    t4 = pd.DataFrame(sum([fmt(m) for m in main_models], []))
    t6 = pd.DataFrame(sum([fmt(m) for m in abl_models], []))
    t4.to_csv(os.path.join(a.out, "metrics_models_by_shift.csv"), index=False)
    t6.to_csv(os.path.join(a.out, "metrics_ablation_by_shift.csv"), index=False)
    # compact tables: whole test period (both shifts), mean (± SD over seeds)
    t4c = pd.DataFrame(sum([fmt(m, shifts=("All",)) for m in main_models], [])).drop(columns="Shift")
    t4c.to_csv(os.path.join(a.out, "metrics_models.csv"), index=False)
    full = agg.loc[("Hybrid LSTM-XGBoost", "All"), ("RMSE", "mean")]
    t6c = pd.DataFrame(sum([fmt(m, shifts=("All",)) for m in abl_models], [])).drop(columns=["Shift", "MAPE"])
    t6c["Change in RMSE (%)"] = [round((agg.loc[(m, "All"), ("RMSE", "mean")] - full) / full * 100, 1) for m in abl_models]
    t6c.to_csv(os.path.join(a.out, "metrics_ablation.csv"), index=False)
    log("\nModel comparison - whole test period, mean ± SD over seeds (RMSE, MAE in kWh; MAPE in %)")
    log(t4c.to_string(index=False))
    log("\nAblation - whole test period")
    log(t6c.to_string(index=False))
    log("\nModel comparison by shift")
    log(t4.to_string(index=False))
    log("\nAblation by shift")
    log(t6.to_string(index=False))
    log("Note: XGBoost and the naive models are deterministic, so their SD is zero.")

    # ------------------------------------------------ significance tests
    log("\n[2] Diebold-Mariano and Wilcoxon tests: hybrid vs competitor (test period)")
    rows = []
    for sh, g in (("All", np.ones(len(D.y), bool)), ("Day", D.day), ("Night", ~D.day)):
        m = D.test & g
        for c in ["Persistence (lag-1)", "Seasonal naive (lag-24)", "LSTM", "XGBoost", "Chen et al. weighting"]:
            e1, e2 = errs[SEEDS[0]]["Hybrid LSTM-XGBoost"][m], errs[SEEDS[0]][c][m]
            dm, p = dm_test(e1, e2)
            dmh, ph = dm_test(e1, e2, hac_lags=24)
            w = stats.wilcoxon(np.abs(e1), np.abs(e2))
            nsig = sum(dm_test(errs[s]["Hybrid LSTM-XGBoost"][m], errs[s][c][m], hac_lags=24)[1] < 0.05 for s in SEEDS)
            rows.append(dict(Shift=sh, Competitor=c, n=int(m.sum()), DM=round(dm, 3), p_DM=p,
                             DM_HAC24=round(dmh, 3), p_DM_HAC24=ph, Wilcoxon_p=float(w.pvalue),
                             seeds_with_p_below_005=f"{nsig}/{len(SEEDS)}"))
    t7 = pd.DataFrame(rows)
    t7.to_csv(os.path.join(a.out, "significance_tests.csv"), index=False)
    log(t7.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    log("Negative DM: the hybrid has the smaller squared error. Statistics shown for seed "
        f"{SEEDS[0]}; last column counts seeds with HAC p < 0.05.")

    # ------------------------------------------------ time-ordered CV
    log("\n[3] Expanding-window time-series CV for the XGBoost meta-learner (training period only)")
    X = D.df[feats].copy()
    X["lstm_pred"] = keep["lstm"]
    tuned = np.zeros(len(D.y))
    cv_rows = []
    for sh, g in (("Day", D.day), ("Night", ~D.day)):
        gs = GridSearchCV(XGBRegressor(random_state=42, n_jobs=-1), CV_GRID, cv=TimeSeriesSplit(n_splits=5),
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
        gs.fit(X[D.train & g], D.y[D.train & g])
        tuned[g] = gs.best_estimator_.predict(X[g])
        cv_rows.append(dict(Shift=sh, **gs.best_params_, cv_RMSE=round(-gs.best_score_, 2)))
        log(f"  {sh}: best {gs.best_params_}, CV RMSE {-gs.best_score_:.2f}")
    pd.DataFrame(cv_rows).to_csv(os.path.join(a.out, "timeseries_cv.csv"), index=False)
    log(shift_table(D, {"Hybrid, fixed settings": keep["hybrid"], "Hybrid, CV-selected": tuned},
                    D.test).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

    # CV-selected settings repeated over all seeds (same LSTM predictions as the main runs)
    best = {r["Shift"]: {k: r[k] for k in CV_GRID} for r in cv_rows}
    cv_tabs, cv_rows_dm = [], []
    for seed in SEEDS:
        Xs = D.df[feats].copy()
        Xs["lstm_pred"] = lp_seed[seed]
        hy_cv = np.zeros(len(D.y))
        for sh, g in (("Day", D.day), ("Night", ~D.day)):
            m = XGBRegressor(random_state=seed, n_jobs=-1, **best[sh])
            m.fit(Xs[D.train & g], D.y[D.train & g])
            hy_cv[g] = m.predict(Xs[g])
        t = shift_table(D, {"Hybrid, CV-selected": hy_cv}, D.test)
        t["seed"] = seed
        cv_tabs.append(t)
        mt = D.test
        e_cv = (D.y - hy_cv)[mt]
        for comp in ("LSTM", "Hybrid LSTM-XGBoost"):
            dm, p = dm_test(e_cv, errs[seed][comp][mt], hac_lags=24)
            cv_rows_dm.append(dict(seed=seed, Competitor=comp, DM_HAC24=round(dm, 3), p_DM_HAC24=p))
    cvs = pd.concat(cv_tabs)
    cvs.to_csv(os.path.join(a.out, "timeseries_cv_all_seeds.csv"), index=False)
    cvd = pd.DataFrame(cv_rows_dm)
    cvd.to_csv(os.path.join(a.out, "timeseries_cv_dm.csv"), index=False)
    cva = cvs.groupby("Shift", sort=False)[["RMSE", "MAE", "MAPE", "R2"]].agg(["mean", "std"]).fillna(0.0)
    log("\n  CV-selected settings over seeds " + str(SEEDS) + " (mean ± SD, test period)")
    for sh, r in cva.iterrows():
        log(f"    {sh:5s} RMSE {r[('RMSE','mean')]:.2f} ± {r[('RMSE','std')]:.2f}  MAE {r[('MAE','mean')]:.2f} ± {r[('MAE','std')]:.2f}"
            f"  MAPE {r[('MAPE','mean')]:.2f} ± {r[('MAPE','std')]:.2f}  R2 {r[('R2','mean')]:.3f} ± {r[('R2','std')]:.3f}")
    for comp in ("LSTM", "Hybrid LSTM-XGBoost"):
        sub = cvd[cvd.Competitor == comp]
        nneg = int(((sub.p_DM_HAC24 < 0.05) & (sub.DM_HAC24 < 0)).sum())
        npos = int(((sub.p_DM_HAC24 < 0.05) & (sub.DM_HAC24 > 0)).sum())
        log(f"    DM (HAC 24) CV-selected hybrid vs {comp}: significantly lower error in {nneg}/{len(SEEDS)} seeds, "
            f"significantly higher in {npos}/{len(SEEDS)} seeds")

    # ------------------------------------------------ imputation sensitivity
    if missing.sum() > 0:
        log("\n[4] Imputation sensitivity (seed 42)")
        d2, cut2, _ = load_plant_file(a.data, impute="interpolate")
        D2 = Data(d2, cut2)
        h2 = xgb_pred(D2, feats, 42, meta=lstm_pred(D2, 42))
        log(shift_table(D2, {"Hybrid, time interpolation": h2}, D2.test).to_string(index=False))
    else:
        log("\n[4] No missing values in the input file; imputation has no effect.")

    # ------------------------------------------------ threshold and DSM on measured test load
    log("\n[5] Peak-load threshold and reduction rule (measured load, test period)")
    on_tr = D.train & (D.df["on_peak"].values == 1)
    on_te = D.test & (D.df["on_peak"].values == 1)
    pct = stats.percentileofscore(D.y[on_tr], THRESHOLD_KWH)
    log(f"  {THRESHOLD_KWH:,.0f} kWh = {pct:.1f}th percentile of training weekday On-Peak load")
    log(f"  Share of test weekday On-Peak hours above threshold: {(D.y[on_te] > THRESHOLD_KWH).mean() * 100:.1f} %")
    y, f = D.y, keep["hybrid"]
    dates = D.df["dt"].dt.date.values
    base_peak = pd.Series(y[D.test]).groupby(dates[D.test]).max()
    dsm_rows = []
    for alpha in ALPHAS:
        for trig, lab in ((y > THRESHOLD_KWH, "measured exceeds threshold"),
                          (f > THRESHOLD_KWH, "forecast exceeds threshold")):
            flag = on_te & trig
            adj = np.where(flag, y * (1 - alpha), y)
            red = (y - adj)[D.test].sum()
            new_peak = pd.Series(adj[D.test]).groupby(dates[D.test]).max()
            dsm_rows.append(dict(alpha=alpha, trigger=lab, flagged_hours=int(flag.sum()),
                                 total_kWh=round(y[D.test].sum(), 1), reduction_kWh=round(red, 1),
                                 reduction_pct=round(red / y[D.test].sum() * 100, 2),
                                 mean_daily_peak_reduction_pct=round(((base_peak - new_peak) / base_peak).mean() * 100, 2),
                                 saving_THB=round(red * TARIFF_THB)))
    t9 = pd.DataFrame(dsm_rows)
    t9.to_csv(os.path.join(a.out, "peak_reduction_test.csv"), index=False)
    log(t9.to_string(index=False))
    T, Pf = y[on_te] > THRESHOLD_KWH, f[on_te] > THRESHOLD_KWH
    tp, fp, fn, tn = int((T & Pf).sum()), int((~T & Pf).sum()), int((T & ~Pf).sum()), int((~T & ~Pf).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    t8 = dict(TP=tp, FP=fp, FN=fn, TN=tn, precision=round(prec, 3), recall=round(rec, 3),
              F1=round(2 * prec * rec / (prec + rec), 3) if tp else float("nan"),
              accuracy=round((tp + tn) / len(T), 3), false_alarm_rate=round(fp / (fp + tn), 3) if fp + tn else float("nan"))
    pd.DataFrame([t8]).to_csv(os.path.join(a.out, "exceedance_flagging.csv"), index=False)
    log("  Forecast flagging of exceedances (test weekday On-Peak hours): " + json.dumps(t8))

    # ------------------------------------------------ figures
    figures(D, keep, a.out, log)
    log(f"\nDone in {(time.time() - t0) / 60:.1f} min. Outputs in {a.out}/")
    return dict(D=D, keep=keep, feats=feats, agg=agg)


def figures(D, keep, out, log):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    tr = D.df[D.train]
    wk = tr[tr["dow"] < 5]
    g = wk.groupby("hour")["EN"]
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axvspan(9, 22, color="0.92", label="Weekday On-Peak (09:00-22:00)")
    ax.fill_between(g.mean().index, g.quantile(0.1), g.quantile(0.9), alpha=0.25, label="10th-90th percentile")
    ax.plot(g.mean().index, g.mean().values, marker="o", ms=3, label="Weekday mean")
    ax.axhline(THRESHOLD_KWH, color="tab:red", lw=1, label=f"Threshold {THRESHOLD_KWH:,.0f} kWh")
    ax.set_xlabel("Hour of day"); ax.set_ylabel("Hourly consumption (kWh)"); ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(out, "training_profile.png"), dpi=300); plt.close(fig)

    te = D.test
    for sh, gm in (("day", D.day), ("night", ~D.day)):
        m = te & gm
        fig, ax = plt.subplots(figsize=(10, 3.2))
        ax.plot(D.df["dt"][m], D.y[m], lw=0.7, label="Measured")
        ax.plot(D.df["dt"][m], keep["hybrid"][m], lw=0.7, label="Hybrid forecast")
        ax.set_ylabel("kWh"); ax.legend(fontsize=7); fig.tight_layout()
        fig.savefig(os.path.join(out, f"test_{sh}.png"), dpi=300); plt.close(fig)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(D.y[te], keep["hybrid"][te], s=4, alpha=0.4)
    lim = [0, max(D.y[te].max(), keep["hybrid"][te].max()) * 1.05]
    ax.plot(lim, lim, "k--", lw=1, label="y = x")
    ax.axhline(D.y[D.train].max(), color="tab:red", lw=1, label="Training maximum")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel("Measured (kWh)"); ax.set_ylabel("Hybrid forecast (kWh)")
    ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(out, "scatter_test_quick.png"), dpi=300); plt.close(fig)
    log("  Figures written.")

In [ ]:
RES = main(DATA, OUT, QUICK)

In [ ]:
from IPython.display import display, Image
for name in ["metrics_models", "metrics_ablation", "significance_tests", "timeseries_cv", "exceedance_flagging", "peak_reduction_test"]:
    print("\n" + name); display(pd.read_csv(os.path.join(OUT, name + ".csv")))

## 9. Peak-load reduction scenario

In [ ]:
def pr_evaluate(df, alphas, th, tariff_on, tariff_off):
    on = df["on_peak"].values == 1
    y, f = df["measured"].values, df["forecast"].values
    dates = df["dt"].dt.date.values
    months = df["dt"].dt.to_period("M").values
    base_daily_peak = pd.Series(y).groupby(dates).max()
    base_monthly_max = pd.Series(y).groupby(months).max()
    cost = np.where(on, tariff_on, tariff_off)
    rows, monthly = [], []
    for a in alphas:
        for trig_name, trig in (("forecast", f > th), ("oracle (measured)", y > th)):
            flag = on & trig
            adj = np.where(flag, y * (1 - a), y)
            red = y - adj
            dp = pd.Series(adj).groupby(dates).max()
            mm = pd.Series(adj).groupby(months).max()
            rows.append(dict(
                alpha=a, trigger=trig_name, flagged_hours=int(flag.sum()),
                flagged_days=int(pd.Series(flag).groupby(dates).any().sum()),
                baseline_kWh=round(y.sum(), 1), scenario_kWh=round(adj.sum(), 1),
                reduction_kWh=round(red.sum(), 1), reduction_pct=round(red.sum() / y.sum() * 100, 2),
                onpeak_reduction_pct=round(red[on].sum() / y[on].sum() * 100, 2),
                mean_daily_peak_reduction_pct=round(((base_daily_peak - dp) / base_daily_peak).mean() * 100, 2),
                mean_monthly_max_reduction_pct=round(((base_monthly_max - mm) / base_monthly_max).mean() * 100, 2),
                energy_cost_saving_THB=round((red * cost).sum()),
            ))
            if trig_name == "forecast":
                m = pd.DataFrame({"month": months, "base": y, "adj": adj}).groupby("month").sum()
                m["alpha"] = a
                m["reduction_kWh"] = m["base"] - m["adj"]
                m["reduction_pct"] = m["reduction_kWh"] / m["base"] * 100
                monthly.append(m.reset_index())
    return pd.DataFrame(rows), pd.concat(monthly)


def pr_flagging(df, th):
    on = df["on_peak"].values == 1
    T = df["measured"].values[on] > th
    P = df["forecast"].values[on] > th
    tp, fp = int((T & P).sum()), int((~T & P).sum())
    fn, tn = int((T & ~P).sum()), int((~T & ~P).sum())
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(onpeak_hours=int(on.sum()), TP=tp, FP=fp, FN=fn, TN=tn, precision=round(prec, 3),
                recall=round(rec, 3), F1=round(2 * prec * rec / max(prec + rec, 1e-9), 3),
                accuracy=round((tp + tn) / max(len(T), 1), 3), false_alarm_rate=round(fp / max(fp + tn, 1), 3))


def threshold_sensitivity(df, y_train_onpeak, alpha=0.10,
                          percentiles=(55, 60, 64.4, 70, 75), fixed=(2000, 2100, 2200, 2300, 2400)):
    """Repeat the rule for several thresholds (training percentiles and fixed values)."""
    on = df["on_peak"].values == 1
    y, f = df["measured"].values, df["forecast"].values
    dates = df["dt"].dt.date.values
    bp = pd.Series(y).groupby(dates).max()
    ths = sorted(set([round(float(np.percentile(y_train_onpeak, p))) for p in percentiles] + list(fixed)))
    rows = []
    for th in ths:
        T, P = y[on] > th, f[on] > th
        tp, fp, fn = int((T & P).sum()), int((~T & P).sum()), int((T & ~P).sum())
        flag = on & (f > th)
        adj = np.where(flag, y * (1 - alpha), y)
        npk = pd.Series(adj).groupby(dates).max()
        rows.append(dict(threshold_kWh=th, train_percentile=round(stats.percentileofscore(y_train_onpeak, th), 1),
                         flagged_hours=int(flag.sum()), reduction_pct=round((y - adj).sum() / y.sum() * 100, 2),
                         mean_daily_peak_reduction_pct=round(((bp - npk) / bp).mean() * 100, 2),
                         precision=round(tp / max(tp + fp, 1), 3), recall=round(tp / max(tp + fn, 1), 3)))
    return pd.DataFrame(rows)


def pr_figures(df, th, alpha, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    on = df["on_peak"].values == 1
    flag = on & (df["forecast"].values > th)
    df = df.assign(scenario=np.where(flag, df["measured"] * (1 - alpha), df["measured"]))
    wk = df[df["dt"].dt.dayofweek < 5]
    prof = wk.groupby(wk["dt"].dt.hour)[["measured", "forecast", "scenario"]].mean()
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.axvspan(9, 22, color="0.93", label="On-Peak 09:00-22:00")
    ax.plot(prof.index, prof["measured"], "o-", ms=3, label="Measured (baseline)")
    ax.plot(prof.index, prof["forecast"], "s-.", ms=3, label="Hybrid forecast")
    ax.plot(prof.index, prof["scenario"], "^--", ms=3, label=f"Peak-load reduction scenario ({int(alpha * 100)} %)")
    ax.axhline(th, color="tab:red", lw=0.8, label=f"Threshold {th:,.0f} kWh")
    ax.set_xlabel("Hour of day"); ax.set_ylabel("Mean weekday consumption (kWh)")
    ax.set_xticks(range(0, 24, 2)); ax.legend(fontsize=7, ncol=2); fig.tight_layout()
    fig.savefig(os.path.join(out, "peak_reduction_weekday_profile.png"), dpi=300); plt.close(fig)

    daily = df.groupby(df["dt"].dt.date).agg(base=("measured", "max"), s1=("scenario", "max"))
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(pd.to_datetime(daily.index), daily["base"], lw=0.9, label="Baseline daily peak")
    ax.plot(pd.to_datetime(daily.index), daily["s1"], lw=0.9, ls="--", label=f"Peak-load reduction scenario daily peak ({int(alpha * 100)} %)")
    ax.set_ylabel("Daily peak (kWh)"); ax.legend(fontsize=7); fig.tight_layout()
    fig.savefig(os.path.join(out, "peak_reduction_daily_peak.png"), dpi=300); plt.close(fig)

    # example day: weekday with the most flagged hours
    fd = pd.Series(flag).groupby(df["dt"].dt.date.values).sum()
    day = fd.idxmax()
    d = df[df["dt"].dt.date == day]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.plot(d["dt"].dt.hour, d["measured"], "o-", ms=3, label="Measured")
    ax.plot(d["dt"].dt.hour, d["forecast"], "s-.", ms=3, label="Hybrid forecast")
    ax.plot(d["dt"].dt.hour, d["scenario"], "^--", ms=3, label=f"Peak-load reduction scenario ({int(alpha * 100)} %)")
    ax.axhline(th, color="tab:red", lw=0.8)
    ax.set_title(f"Example day {day}", fontsize=9); ax.set_xlabel("Hour"); ax.set_ylabel("kWh")
    ax.legend(fontsize=7); fig.tight_layout()
    fig.savefig(os.path.join(out, "peak_reduction_example_day.png"), dpi=300); plt.close(fig)
    return day


def peak_reduction(data, out, threshold=THRESHOLD_KWH, alphas="0.05,0.10,0.15", tariff_on=TARIFF_THB, tariff_off=TARIFF_THB, seed=42, quick=False):
    global LSTM_EPOCHS
    a = argparse.Namespace(data=data, out=out, threshold=threshold, alphas=alphas, tariff_on=tariff_on, tariff_off=tariff_off, seed=seed, quick=quick)
    if a.quick:
        LSTM_EPOCHS = 2
    os.makedirs(a.out, exist_ok=True)
    alphas = [float(x) for x in a.alphas.split(",")]

    d, cut, _ = load_plant_file(a.data)
    D = Data(d, cut)
    lp = lstm_pred(D, a.seed)
    fc = xgb_pred(D, FEATURES, a.seed, meta=lp)
    te = D.test
    df = pd.DataFrame({"dt": D.df["dt"][te].values, "measured": D.y[te], "forecast": fc[te],
                       "on_peak": D.df["on_peak"][te].values, "shift": np.where(D.day[te], "day", "night")})
    df.to_csv(os.path.join(a.out, "peak_reduction_hourly.csv"), index=False)

    on_tr = D.train & (D.df["on_peak"].values == 1)
    pct = stats.percentileofscore(D.y[on_tr], a.threshold)
    summ, monthly = pr_evaluate(df, alphas, a.threshold, a.tariff_on, a.tariff_off)
    fl = pr_flagging(df, a.threshold)
    summ.to_csv(os.path.join(a.out, "peak_reduction_summary.csv"), index=False)
    monthly.to_csv(os.path.join(a.out, "peak_reduction_monthly.csv"), index=False)
    pd.DataFrame([fl]).to_csv(os.path.join(a.out, "peak_reduction_flagging.csv"), index=False)
    sens = threshold_sensitivity(df, D.y[on_tr], alpha=0.10)
    sens.to_csv(os.path.join(a.out, "peak_reduction_threshold_sensitivity.csv"), index=False)
    day = pr_figures(df, a.threshold, 0.10, a.out)

    with open(os.path.join(a.out, "peak_reduction_report.txt"), "w", encoding="utf-8") as f:
        def w(s=""):
            print(s); f.write(s + "\n")
        w("Peak-load reduction scenario - forecast-triggered peak-load reduction")
        w(f"Test period: {df.dt.min()} to {df.dt.max()} ({len(df)} hours)")
        w(f"Threshold {a.threshold:,.0f} kWh = {pct:.1f}th percentile of training weekday On-Peak load")
        w(f"Tariff: On-Peak {a.tariff_on} THB/kWh, Off-Peak {a.tariff_off} THB/kWh (energy charge only)")
        w("\n" + summ.to_string(index=False))
        w("\nForecast flagging of threshold exceedances (weekday On-Peak hours):")
        w(str(fl))
        w("\nThreshold sensitivity (alpha = 0.10, forecast trigger):")
        w(sens.to_string(index=False))
        w(f"\nExample day in peak_reduction_example_day.png: {day}")
        w("\nMonthly results (forecast trigger):")
        w(monthly.round(2).to_string(index=False))

In [ ]:
peak_reduction(DATA, OUT_PEAK, threshold=THRESHOLD, quick=QUICK)
for name in ["peak_reduction_summary", "peak_reduction_flagging", "peak_reduction_threshold_sensitivity"]:
    print("\n" + name); display(pd.read_csv(os.path.join(OUT_PEAK, name + ".csv")))

## 10. Daily check against utility meter registers (optional)

In [ ]:
def read_reports(pattern):
    import openpyxl
    rows = []
    for f in sorted(glob.glob(pattern)):
        ws = openpyxl.load_workbook(f, data_only=True).active
        for r in ws.iter_rows(min_row=1, values_only=True):
            first = str(r[0]).strip() if r[0] is not None else ""
            if len(first) == 10 and first[2] == "/" and first[5] == "/":
                rows.append(dict(date=pd.to_datetime(first, dayfirst=True), kWh_rateA=r[1], kWh_rateB=r[2],
                                 kWh_rateC=r[3], kWh_total=r[4], source_file=os.path.basename(f)))
    return pd.DataFrame(rows).set_index("date").sort_index()


def load_level(daily, dev_path):
    span = f"{daily.index.min():%b %Y} to {daily.index.max():%b %Y}"
    out = {f"Meter reports ({span}) mean daily kWh": daily["kWh_total"].mean()}
    if dev_path:
        d = pd.read_csv(dev_path, encoding="utf-8-sig")
        d.columns = [c.strip() for c in d.columns]
        d = d.dropna(subset=["timestamp"])
        d["dt"] = pd.to_datetime(d["timestamp"].astype(str).str.strip(), dayfirst=True, errors="coerce")
        g = d.groupby(d["dt"].dt.date)["EN"].agg(["sum", "size"])
        full = g[g["size"] == 24]["sum"]
        out["2022-2023 mean daily kWh (complete days)"] = full.mean()
        out["2022-2023 training-period max hourly kWh"] = d["EN"].max()
        out["ratio meter reports / 2022-2023 (mean daily)"] = daily["kWh_total"].mean() / full.mean()
    return pd.Series(out).round(3)


def peak_reduction_daily(daily, percentile, alphas):
    wk = daily[daily["kWh_rateA"].notna() & (daily["kWh_rateA"] > 0)]
    th = np.percentile(wk["kWh_rateA"], percentile)
    rows = []
    for a in alphas:
        flag = wk["kWh_rateA"] > th
        red = (wk["kWh_rateA"][flag] * a).sum()
        rows.append(dict(alpha=a, percentile=percentile, threshold_onpeak_kWh_per_day=round(th),
                         weekdays=len(wk), flagged_days=int(flag.sum()), reduction_kWh=round(red),
                         reduction_pct_of_total=round(red / daily["kWh_total"].sum() * 100, 2),
                         reduction_pct_of_onpeak=round(red / wk["kWh_rateA"].sum() * 100, 2)))
    return pd.DataFrame(rows)


def trial_day(daily, prod, date):
    date = pd.Timestamp(date)
    m = daily.loc[str(date.to_period("M"))]
    wk = m[m["kWh_rateA"].notna() & (m["kWh_rateA"] > 0)].copy()
    if prod is not None:
        wk = wk.join(prod[["total_output_kg"]], how="left")
        wk["kWh_per_kg"] = wk["kWh_total"] / wk["total_output_kg"].replace(0, np.nan)
        wk["onpeak_kWh_per_kg"] = wk["kWh_rateA"] / wk["total_output_kg"].replace(0, np.nan)
    res = {"trial date": str(date.date()), "weekdays in month": len(wk),
           "trial On-Peak kWh": wk.loc[date, "kWh_rateA"], "other weekdays mean On-Peak kWh": wk.drop(date)["kWh_rateA"].mean(),
           "trial rank by On-Peak kWh (1 = highest)": int((wk["kWh_rateA"] > wk.loc[date, "kWh_rateA"]).sum() + 1)}
    if prod is not None:
        for c in ("kWh_per_kg", "onpeak_kWh_per_kg"):
            res[f"trial {c}"] = wk.loc[date, c]
            res[f"other weekdays mean {c}"] = wk.drop(date)[c].mean()
            res[f"trial rank by {c} (1 = highest)"] = int((wk[c] > wk.loc[date, c]).sum() + 1)
        res["trial total output kg"] = wk.loc[date, "total_output_kg"]
        res["other weekdays mean output kg"] = wk.drop(date)["total_output_kg"].mean()
        o = wk.drop(date).dropna(subset=["total_output_kg"])
        for c in ("kWh_total", "kWh_rateA"):
            b = np.polyfit(o["total_output_kg"], o[c], 1)        # energy vs output, other weekdays
            exp = float(np.polyval(b, wk.loc[date, "total_output_kg"]))
            sd = float((o[c] - np.polyval(b, o["total_output_kg"])).std())
            res[f"{c}: expected from output (linear fit, other weekdays)"] = round(exp)
            res[f"{c}: actual minus expected (%)"] = round((wk.loc[date, c] - exp) / exp * 100, 1)
            res[f"{c}: z-score vs residual SD"] = round((wk.loc[date, c] - exp) / sd, 2)
    return pd.Series(res), wk


def daily_forecast_check(daily, prod, holidays_years=(2025,)):
    import holidays
    from sklearn.linear_model import RidgeCV
    from xgboost import XGBRegressor
    d = prod.join(daily[["kWh_total"]], how="inner", rsuffix="_meter")
    d["y"] = d["kWh_total_meter"] if "kWh_total_meter" in d else d["kWh_total"]
    th = holidays.Thailand(years=list(holidays_years))
    d["dow"] = d.index.dayofweek
    d["weekend"] = (d["dow"] >= 5).astype(int)
    d["holiday"] = [int(x.date() in th) for x in d.index]
    d["lag1"], d["lag2"], d["lag7"] = d["y"].shift(1), d["y"].shift(2), d["y"].shift(7)
    d["roll7_prev"] = d["y"].shift(1).rolling(7, min_periods=3).mean()      # previous days only
    d["out_roll7_prev"] = d["total_output_kg"].shift(1).rolling(7, min_periods=3).mean()
    d = d.dropna(subset=["lag7", "roll7_prev", "out_roll7_prev"])
    prod_cols = [c for c in ("paste_output_kg", "packing_output_kg", "freezing_output_kg", "total_output_kg") if c in d]
    F = prod_cols + ["dow", "weekend", "holiday", "lag1", "lag2", "roll7_prev", "out_roll7_prev"]
    cut = int(len(d) * 0.8)
    tr, te = d.iloc[:cut], d.iloc[cut:]

    def met(y, p):
        e = y - p
        return dict(RMSE=round(float(np.sqrt((e ** 2).mean()))), MAE=round(float(np.abs(e).mean())),
                    MAPE=round(float((np.abs(e) / y).mean() * 100), 2),
                    R2=round(float(1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum()), 3))
    mu, sd = tr[F].mean(), tr[F].std().replace(0, 1)
    ridge = RidgeCV(alphas=np.logspace(-2, 4, 30)).fit((tr[F] - mu) / sd, tr["y"])
    xgb = XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42).fit(tr[F], tr["y"])
    rows = [dict(Model="Persistence (previous day)", **met(te["y"], te["lag1"])),
            dict(Model="Naive (same weekday, previous week)", **met(te["y"], te["lag7"])),
            dict(Model="Mean of previous 7 days", **met(te["y"], te["roll7_prev"])),
            dict(Model="Ridge regression (production + lags)", **met(te["y"], ridge.predict((te[F] - mu) / sd))),
            dict(Model="XGBoost (production + lags)", **met(te["y"], xgb.predict(te[F])))]
    span = f"train {tr.index.min().date()} to {tr.index.max().date()} ({len(tr)} days), test {te.index.min().date()} to {te.index.max().date()} ({len(te)} days)"
    return pd.DataFrame(rows), span


def daily_check(out, reports=None, daily_csv=None, dev=None, production=None, trial_date=None,
                percentile=64.4, alphas=(0.05, 0.10, 0.15), forecast_check=False):
    os.makedirs(out, exist_ok=True)
    if reports:
        daily = read_reports(reports)
    else:
        daily = pd.read_csv(daily_csv, parse_dates=["date"]).set_index("date").sort_index()
    for c in ("kWh_rateA", "kWh_rateB", "kWh_rateC", "kWh_total"):
        daily[c] = pd.to_numeric(daily[c], errors="coerce")
    daily.to_csv(os.path.join(out, "meter_daily.csv"))
    prod = None
    if production:
        prod = pd.read_csv(production, parse_dates=["date"]).set_index("date").sort_index()
    lines = []

    def w(s=""):
        print(s)
        lines.append(str(s))
    w("2025 cross-check with utility meter reports (daily TOU registers)")
    w(f"Report days: {len(daily)} ({daily.index.min().date()} to {daily.index.max().date()}), "
      f"total {daily['kWh_total'].sum() / 1e6:.2f} GWh, On-Peak share {daily['kWh_rateA'].sum() / daily['kWh_total'].sum() * 100:.1f} %")

    lv = load_level(daily, dev)
    lv.to_csv(os.path.join(out, "load_level_comparison.csv"), header=["value"])
    w("\n[1] Load level")
    w(lv.to_string())

    s1 = peak_reduction_daily(daily, percentile, alphas)
    s1.to_csv(os.path.join(out, "peak_reduction_daily.csv"), index=False)
    w(f"\n[2] Peak-load reduction scenario at daily level on measured 2025 On-Peak energy (percentile {percentile})")
    w(s1.to_string(index=False))
    sens = pd.concat([peak_reduction_daily(daily, p, [0.10]) for p in (55, 60, 64.4, 70, 75)])
    sens.to_csv(os.path.join(out, "peak_reduction_daily_sensitivity.csv"), index=False)
    w("\n    Percentile sensitivity (alpha = 0.10)")
    w(sens[["percentile", "threshold_onpeak_kWh_per_day", "flagged_days", "reduction_pct_of_total", "reduction_pct_of_onpeak"]].to_string(index=False))

    if trial_date:
        tres, tab = trial_day(daily, prod, trial_date)
        tres.to_csv(os.path.join(out, "trial_day_check.csv"), header=["value"])
        tab.to_csv(os.path.join(out, "trial_month_weekdays.csv"))
        w(f"\n[3] Trial day {trial_date} compared with other weekdays of the month")
        w(tres.to_string())

    if forecast_check and prod is not None:   # exploratory; not part of the main analysis
        fc, span = daily_forecast_check(daily, prod)
        fc.to_csv(os.path.join(out, "daily_forecast_check.csv"), index=False)
        w(f"\n[4] Daily forecasting check within 2025, exploratory, not part of the main analysis ({span})")
        w(fc.to_string(index=False))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        wk = daily[daily["kWh_rateA"].notna() & (daily["kWh_rateA"] > 0)]
        th = np.percentile(wk["kWh_rateA"], percentile)
        fig, ax = plt.subplots(figsize=(9, 3.4))
        ax.plot(wk.index, wk["kWh_rateA"], ".-", ms=3, lw=0.7, label="Measured weekday On-Peak energy")
        ax.plot(wk.index, np.where(wk["kWh_rateA"] > th, wk["kWh_rateA"] * 0.9, wk["kWh_rateA"]), lw=0.7, ls="--",
                label="Peak-load reduction scenario (10 %)")
        ax.axhline(th, color="tab:red", lw=0.8, label=f"{percentile}th percentile ({th:,.0f} kWh/day)")
        if trial_date:
            ax.axvline(pd.Timestamp(trial_date), color="0.5", ls=":", lw=1, label="Trial day")
        ax.set_ylabel("kWh per day"); ax.legend(fontsize=7); fig.tight_layout()
        fig.savefig(os.path.join(out, "daily_onpeak_meter_check.png"), dpi=300); plt.close(fig)
    except Exception as ex:
        w(f"(figure skipped: {ex})")
    open(os.path.join(out, "daily_meter_check_report.txt"), "w", encoding="utf-8").write("\n".join(lines))

In [ ]:
if USE_REPORTS:
    daily_check(OUT_METER, reports=REPORTS, dev=DATA, production=PRODUCTION if os.path.exists(PRODUCTION) else None, trial_date=TRIAL_DATE)
else:
    daily_check(OUT_METER, daily_csv="data/synthetic_daily_meter.csv", dev=DATA, production="data/synthetic_daily_production.csv", trial_date=TRIAL_DATE)

## 11. Figures

In [ ]:
BLUE, ORANGE, MAGENTA, RED = "#1565C0", "#F57C20", "#C2185B", "#E53935"
BAR_DAY, BAR_NIGHT = "#90C2F5", "#F8B26A"
PINK = "#F8D7DA"


def _style():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
        "font.weight": "bold", "axes.labelweight": "bold", "axes.titleweight": "bold",
        "font.size": 12, "axes.linewidth": 1.2, "legend.fontsize": 10,
        "xtick.major.width": 1.0, "ytick.major.width": 1.0, "savefig.dpi": 300,
    })
    return plt


def _save(fig, out, name):
    fig.tight_layout()
    fig.savefig(os.path.join(out, name + ".png"), dpi=300)


def fig_timeseries(D, out, plt):
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(D.full["dt"], D.full["EN"], color=BLUE, lw=0.5)
    cut = D.full["dt"].iloc[int(len(D.full) * 0.8)]
    ax.axvline(cut, color=RED, ls="--", lw=1.2)
    ax.text(cut, ax.get_ylim()[1] * 0.95, "  Test period", color=RED)
    ax.set_xlabel("Date"); ax.set_ylabel("Energy consumption (kWh)")
    _save(fig, out, "time_series"); plt.close(fig)


def fig_actual_vs_pred(D, keep, out, plt, n=300):
    fig, axes = plt.subplots(2, 1, figsize=(10, 7))
    for ax, (lab, g, tag) in zip(axes, (("day", D.day, "(a) Hybrid model performance during day shift"),
                                        ("night", ~D.day, "(b) Hybrid model performance during night shift"))):
        m = np.where(D.test & g)[0][:n]
        ax.plot(range(len(m)), D.y[m], color=BLUE, lw=1.6, label="Actual")
        ax.plot(range(len(m)), keep["hybrid"][m], color=ORANGE, lw=1.6, ls="--", label="Predicted (Hybrid)")
        ax.set_xlabel("Hourly observation in the test period (this shift only)")
        ax.set_ylabel("Energy consumption (kWh)"); ax.legend(loc="upper right")
        ax.set_title(tag, y=-0.33, fontsize=13)
    fig.subplots_adjust(hspace=0.55)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "actual_vs_predicted.png"), dpi=300)
    plt.close(fig)


def fig_metric_bars(agg, out, plt, models=("LSTM", "XGBoost", "Hybrid LSTM-XGBoost")):
    labels = ["LSTM", "XGBoost", "Hybrid"]
    fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
    for ax, (met, title) in zip(axes.ravel(), (("RMSE", "RMSE"), ("MAE", "MAE"), ("MAPE", "MAPE"), ("R2", "R$^2$"))):
        x = np.arange(len(models))
        d = [agg.loc[(m, "Day"), (met, "mean")] for m in models]
        n = [agg.loc[(m, "Night"), (met, "mean")] for m in models]
        ed = [agg.loc[(m, "Day"), (met, "std")] for m in models]
        en = [agg.loc[(m, "Night"), (met, "std")] for m in models]
        ax.bar(x - 0.18, d, 0.34, yerr=ed, capsize=3, color=BAR_DAY, label="Day")
        ax.bar(x + 0.18, n, 0.34, yerr=en, capsize=3, color=BAR_NIGHT, label="Night")
        ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_title(title, fontsize=15)
        if met == "R2":
            ax.set_ylim(0, 1.0)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=2, frameon=False, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig(os.path.join(out, "metric_comparison.png"), dpi=300)
    plt.close(fig)


NICE = {"RH": "RH", "air_temp": "Air temp", "monthly_avg_temp": "monthly avg temp", "temp_anomaly": "temp anomaly",
        "Process": "process", "Paste": "paste", "Cold": "cold", "hour": "hour", "month": "month", "dow": "day of week",
        "daytime": "daytime", "on_peak": "on_peak", "holiday": "holiday", "autumn": "autumn", "spring": "spring",
        "winter": "winter", "lag1": "lag1", "lag24": "lag24", "lstm_pred": "LSTM prediction"}


def hybrid_gain(D, feats, lstm, seed=42, params=None):
    """Gain importance of the hybrid XGBoost meta-learner (mean of day and night models)."""
    from xgboost import XGBRegressor
    X = D.df[feats].copy()
    X["lstm_pred"] = lstm
    tot = pd.Series(0.0, index=X.columns)
    for g in (D.day, ~D.day):
        m = XGBRegressor(random_state=seed, n_jobs=-1, n_estimators=100, max_depth=6, learning_rate=0.10,
                         **(params or {}))
        m.fit(X[D.train & g], D.y[D.train & g])
        gain = pd.Series(m.get_booster().get_score(importance_type="gain"))
        tot = tot.add(gain.reindex(X.columns).fillna(0) / 2, fill_value=0)
    return tot.sort_values(ascending=False)


def fig_importance(gain, out, plt, top=12):
    g = gain.head(top)[::-1]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh([NICE.get(k, k) for k in g.index], g.values, color="#64B5F6")
    ax.set_xlabel("Importance Score (gain)"); ax.set_ylabel("Features")
    _save(fig, out, "feature_importance"); plt.close(fig)


def fig_threshold_profile(D, out, plt, th=2200.0):
    tr = D.df[D.train]
    wk = tr[tr["dow"] < 5]
    g = wk.groupby("hour")["EN"]
    mean, lo, hi = g.mean(), g.quantile(0.10), g.quantile(0.90)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.axvspan(9, 22, color=PINK, alpha=0.8, lw=0)
    ax.fill_between(mean.index, lo, hi, color=BLUE, alpha=0.12, lw=0, label="10th–90th percentile")
    ax.plot(mean.index, mean.values, "o-", color=BLUE, lw=1.8, ms=5, label="Mean weekday energy consumption (kWh)")
    ax.axhline(th, color=RED, ls="--", lw=1.4)
    ax.text(0.5, th * 1.01, f"Threshold = {th:,.0f} kWh", color=RED, fontsize=11)
    share = g.apply(lambda v: (v > th).mean() * 100)
    ax.text(15.5, ax.get_ylim()[0] + 0.06 * (ax.get_ylim()[1] - ax.get_ylim()[0]), "On-Peak", color="#3A1B8C", fontsize=14, ha="center")
    ax.set_xticks(range(24)); ax.set_xlabel("Hour of Day"); ax.set_ylabel("Energy consumption (kWh)")
    ax.legend(loc="upper right")
    _save(fig, out, "threshold_profile"); plt.close(fig)
    return share


def _scenario(D, keep, th, alpha):
    te = D.test
    df = pd.DataFrame({"dt": D.df["dt"][te].values, "y": D.y[te], "f": keep["hybrid"][te],
                       "on": D.df["on_peak"][te].values == 1})
    df["s1"] = np.where(df["on"] & (df["f"] > th), df["y"] * (1 - alpha), df["y"])
    return df


def fig_hourly_reduction(D, keep, out, plt, th=2200.0, alpha=0.10):
    df = _scenario(D, keep, th, alpha)
    wk = df[pd.to_datetime(df["dt"]).dt.dayofweek < 5]
    p = wk.groupby(pd.to_datetime(wk["dt"]).dt.hour)[["y", "s1"]].mean()
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axvspan(9, 22, color=PINK, alpha=0.8, lw=0)
    ax.plot(p.index, p["y"], "o-", color=BLUE, lw=1.8, ms=5, label="Baseline (measured)")
    ax.plot(p.index, p["s1"], "o:", color=MAGENTA, lw=1.8, ms=5, label=f"After {int(alpha * 100)}% peak reduction")
    ax.axhline(th, color=RED, ls="--", lw=1.4)
    ax.text(0.3, th + 8, f"Threshold = {th:,.0f} kWh", color=RED, fontsize=11)
    ax.text(15.5, ax.get_ylim()[0] + 0.08 * (ax.get_ylim()[1] - ax.get_ylim()[0]), "On-peak", color="#3A1B8C", fontsize=14, ha="center")
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, max(hi, th + 0.08 * (th - lo)))
    ax.set_xticks(range(24)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Energy consumption (kWh)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=2)
    _save(fig, out, "hourly_peak_reduction"); plt.close(fig)


def fig_example_day(df, out, plt, day=None, th=2200.0, alpha=0.10):
    """One weekday of the test period: measured load, hybrid forecast and load after the rule.
    df needs columns dt, y (measured), f (hybrid forecast), on (weekday On-Peak flag)."""
    df = df.copy()
    df["dt"] = pd.to_datetime(df["dt"])
    if day is None:   # first test day on which the rule flags 4-9 On-Peak hours (else the day with most flags)
        n = df.assign(f=df["on"] & (df["f"] > th)).groupby(df["dt"].dt.date)["f"].sum()
        cand = n[(n >= 4) & (n <= 9)]
        day = (cand.index[0] if len(cand) else n.idxmax())
    g = df[df["dt"].dt.date == pd.Timestamp(day).date()].sort_values("dt")
    if g.empty:
        return None
    h = g["dt"].dt.hour.values
    flag = (g["on"].values) & (g["f"].values > th)
    s1 = np.where(flag, g["y"].values * (1 - alpha), g["y"].values)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axvspan(9, 22, color=PINK, alpha=0.8, lw=0)
    ax.plot(h, g["y"], "o-", color=BLUE, lw=1.8, ms=5, label="Measured load")
    ax.plot(h, g["f"], "s--", color=ORANGE, lw=1.5, ms=4, label="Hybrid forecast")
    ax.plot(h, s1, "o:", color=MAGENTA, lw=1.8, ms=5, label=f"After {int(alpha * 100)}% reduction")
    ax.scatter(h[flag], s1[flag], s=90, facecolors="none", edgecolors=MAGENTA, lw=1.4, zorder=5, label="Flagged hour (forecast > threshold)")
    ax.axhline(th, color=RED, ls="--", lw=1.4)
    ax.text(0.2, th + 0.035 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"Threshold = {th:,.0f} kWh", color=RED, fontsize=11)
    lo, hi = ax.get_ylim()
    ax.text(15.5, lo + 0.06 * (hi - lo), "On-Peak", color="#3A1B8C", fontsize=14, ha="center")
    ax.set_xticks(range(24)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Energy consumption (kWh)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=2)
    fig.tight_layout()
    _save(fig, out, "example_day"); plt.close(fig)
    return int(flag.sum())


def fig_daily_meter(daily_csv, out, plt, percentile=64.4, alpha=0.10, trial_date=None):
    """2025 utility meter: weekday On-Peak energy per day with the daily Peak-load reduction scenario rule."""
    if daily_csv is None or not os.path.exists(daily_csv):
        return None
    d = pd.read_csv(daily_csv, parse_dates=["date"]).set_index("date").sort_index()
    wk = d[d["kWh_rateA"].notna() & (d["kWh_rateA"] > 0)]
    th = np.percentile(wk["kWh_rateA"], percentile)
    s1 = np.where(wk["kWh_rateA"] > th, wk["kWh_rateA"] * (1 - alpha), wk["kWh_rateA"])
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(wk.index, wk["kWh_rateA"], color=BLUE, lw=1.3, label="Measured On-Peak energy (utility meter)")
    ax.plot(wk.index, s1, color=ORANGE, lw=1.3, ls="--", label=f"Peak-load reduction ({int(alpha * 100)}%)")
    ax.axhline(th, color=RED, ls="--", lw=1.2)
    ax.text(wk.index[0], th * 1.01, f"  {percentile}th percentile = {th:,.0f} kWh/day", color=RED, fontsize=10)
    red = (wk["kWh_rateA"] - s1).sum()
    ax.text(0.02, 0.95, f"On-Peak reduction = {red / wk['kWh_rateA'].sum() * 100:.2f}%  |  total = {red / d['kWh_total'].sum() * 100:.2f}%",
            transform=ax.transAxes, va="top", fontsize=11)
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo - 0.25 * (hi - lo), hi)
    ax.set_xlabel("Date"); ax.set_ylabel("Weekday On-Peak energy (kWh/day)")
    ax.legend(loc="lower right")
    _save(fig, out, "daily_onpeak_meter"); plt.close(fig)
    return th


def fig_scatter(D, keep, out, plt):
    te = D.test
    y, p = D.y[te], keep["hybrid"][te]
    lim = [0, max(y.max(), p.max(), D.y[D.train].max()) * 1.05]
    fig, ax = plt.subplots(figsize=(5.6, 5.4))
    ax.scatter(y, p, s=8, alpha=0.45, color=BLUE, edgecolors="none")
    ax.plot(lim, lim, "k--", lw=1.2, label="y = x")
    ax.axhline(D.y[D.train].max(), color=RED, lw=1, ls=":", label=f"Training maximum ({D.y[D.train].max():,.0f} kWh)")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Measured energy consumption (kWh)"); ax.set_ylabel("Predicted energy consumption (kWh)")
    ax.legend(loc="upper left", fontsize=9)
    _save(fig, out, "scatter_test"); plt.close(fig)


def fig_daily_peak(D, keep, out, plt, th=2200.0, alpha=0.10):
    df = _scenario(D, keep, th, alpha)
    day = pd.to_datetime(df["dt"]).dt.date
    dp = df.groupby(day).agg(base=("y", "max"), s1=("s1", "max"))
    red = ((dp["base"] - dp["s1"]) / dp["base"]).mean() * 100
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(pd.to_datetime(dp.index), dp["base"], color=BLUE, lw=1.2, label="Baseline operation")
    ax.plot(pd.to_datetime(dp.index), dp["s1"], color=ORANGE, lw=1.2, ls="--", label="Peak-load reduction strategy")
    ax.text(0.02, 0.95, f"Mean daily peak reduction = {red:.2f}%", transform=ax.transAxes, va="top", fontsize=12)
    ax.set_ylim(0, dp["base"].max() * 1.15)
    ax.set_xlabel("Date"); ax.set_ylabel("Daily peak hourly energy consumption (kWh)")
    ax.legend(loc="upper right")
    _save(fig, out, "daily_peak"); plt.close(fig)
    return red


def make_figures(res, out="figures", threshold=2200.0, alpha=0.10, daily_meter_csv=None):
    os.makedirs(out, exist_ok=True)
    plt = _style()
    D, keep, feats, agg = res["D"], res["keep"], res["feats"], res["agg"]
    fig_timeseries(D, out, plt)
    fig_actual_vs_pred(D, keep, out, plt)
    fig_metric_bars(agg, out, plt)
    gain = hybrid_gain(D, feats, keep["lstm"])
    gain.to_csv(os.path.join(out, "feature_importance_gain.csv"), header=["gain"])
    fig_importance(gain, out, plt)
    share = fig_threshold_profile(D, out, plt, threshold)
    fig_hourly_reduction(D, keep, out, plt, threshold, alpha)
    fig_example_day(_scenario(D, keep, threshold, alpha), out, plt, th=threshold, alpha=alpha)
    fig_daily_meter(daily_meter_csv, out, plt)
    fig_scatter(D, keep, out, plt)
    red = fig_daily_peak(D, keep, out, plt, threshold, alpha)
    print("Figures saved to", out)
    print("Top features (gain):", ", ".join(f"{k} {v:,.0f}" for k, v in gain.head(6).items()))
    print(f"Mean daily peak reduction (test period, {int(alpha * 100)}%): {red:.2f}%")
    return dict(gain=gain, share_above_threshold_by_hour=share, mean_daily_peak_reduction=red)

In [ ]:
FIG = make_figures(RES, out=OUT_FIG, threshold=THRESHOLD, alpha=0.10, daily_meter_csv=os.path.join(OUT_METER, "meter_daily.csv"))
for f in sorted(glob.glob(os.path.join(OUT_FIG, "*.png"))):
    print(os.path.basename(f)); display(Image(f, width=750))

## 12. Download results (Colab)

In [ ]:
import shutil
for d in (OUT, OUT_PEAK, OUT_METER, OUT_FIG):
    shutil.make_archive(d, "zip", d)
if IN_COLAB:
    from google.colab import files
    for d in (OUT, OUT_PEAK, OUT_METER, OUT_FIG):
        files.download(d + ".zip")